# Whisper large-v3 — Malagasy fine-tune (QLoRA, free T4)

Fine-tunes `openai/whisper-large-v3` on `badrex/malagasy-speech-full` using
4-bit QLoRA so the whole thing fits on a free Colab T4 (15 GB VRAM).

**Before running:**
1. `Runtime > Change runtime type > T4 GPU`
2. Edit the **Config** cell (cell 3) to set sample count, output dir, etc.
3. Run all cells in order.

**What gets saved:** a ~50 MB LoRA adapter + processor, either to Google Drive
or your HF Hub repo. The base model weights are never modified.

**VRAM budget on T4 (15 GB):**
- whisper-large-v3 in 4-bit NF4: ~900 MB
- LoRA trainable params (q_proj + v_proj, r=32): ~20 MB
- Activations at batch=2, 30s audio: ~2-4 GB
- Total: ~6-7 GB — fits with headroom


In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No GPU found. Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.34.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" "evaluate>=0.4.0" "jiwer>=3.0.0" "soundfile>=0.12.1"
print("Install complete")

In [ ]:
# ── 3. Config — edit before running ──────────────────────────────────────────

BASE_MODEL   = "openai/whisper-large-v3"      # or whisper-medium / whisper-small
LANGUAGE     = "mg"                            # Malagasy
DATASET_ID   = "badrex/malagasy-speech-full"  # 150h Malagasy, streamed — no 124 GB download

# How many samples to pull from the stream for this run.
# 3000 ~ 30 min training on T4. Increase for a full run (use Kaggle 30h/week).
MAX_SAMPLES  = 3000
EVAL_FRAC    = 0.05   # 5% of MAX_SAMPLES held out for evaluation

# Training knobs
BATCH_SIZE   = 2      # per-device; keep at 2 for T4 to avoid OOM
GRAD_ACCUM   = 8      # effective batch size = BATCH_SIZE * GRAD_ACCUM = 16
LR           = 1e-4
WARMUP_STEPS = 50
EPOCHS       = 3
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05

# Where to save the adapter. Google Drive survives session resets.
OUTPUT_DIR   = "/content/drive/MyDrive/whisper-large-v3-mg-lora"
# Optional: push to HF Hub after training. Leave empty to skip.
HF_HUB_MODEL_ID = ""  # e.g. "YourUsername/whisper-large-v3-malagasy"

print("Config loaded")

In [ ]:
# ── 4. Mount Google Drive (recommended) ──────────────────────────────────────
# Checkpoints saved here survive session resets so you can resume training.
# Skip this cell if you don't need Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── 5. HF login ───────────────────────────────────────────────────────────────
# Required to stream badrex/malagasy-speech-full.
# Get a read token at https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# ── 6. Load processor + model (4-bit QLoRA) ───────────────────────────────────
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"Loading processor for {BASE_MODEL}...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language=LANGUAGE, task="transcribe")

print("Loading model in 4-bit NF4 (QLoRA)...")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # saves ~0.4 bits/param extra
)
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
)

# Pin Malagasy so the decoder doesn't drift back to English during eval
model.generation_config.language = LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False  # required when gradient_checkpointing=True

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print("Model ready")

In [ ]:
# ── 7. Stream + preprocess dataset ────────────────────────────────────────────
# Dataset.from_generator writes one sample at a time to disk via Arrow writer
# (writer_batch_size=100 → ~96 MB peak RAM regardless of MAX_SAMPLES).
import os, shutil
from datasets import load_dataset, Audio, Dataset, Array2D, Sequence, Value, Features
from tqdm.auto import tqdm

CACHE_DIR = "/content/hf_cache"
# Always start fresh — avoids shape-mismatch errors from a previous run.
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR)

# Read actual mel-bin count and frame count from the loaded processor.
# whisper-large-v3 → 128 bins; tiny/base/small/medium → 80 bins.
N_MELS   = processor.feature_extractor.feature_size   # e.g. 128
N_FRAMES = processor.feature_extractor.nb_max_frames  # 3000
print(f"Feature shape: ({N_MELS}, {N_FRAMES})")

# Peek at one sample to detect the text column, then discard the iterator.
_peek = load_dataset(DATASET_ID, split="train", streaming=True, trust_remote_code=True)
_first = next(iter(_peek))
TEXT_COL = next(
    (c for c in ["sentence", "transcription", "text", "transcript"] if c in _first),
    None,
)
if TEXT_COL is None:
    raise ValueError(f"No text column found. Columns: {list(_first.keys())}")
print(f"Text column: '{TEXT_COL}'")
del _peek, _first

def sample_generator():
    stream = load_dataset(DATASET_ID, split="train", streaming=True, trust_remote_code=True)
    stream = stream.cast_column("audio", Audio(sampling_rate=16_000))
    count, skipped = 0, 0
    pbar = tqdm(total=MAX_SAMPLES, desc="Processing")
    for sample in stream:
        if count >= MAX_SAMPLES:
            break
        text = (sample.get(TEXT_COL) or "").strip()
        if not text:
            skipped += 1
            continue
        try:
            audio = sample["audio"]
            feats = processor.feature_extractor(
                audio["array"], sampling_rate=audio["sampling_rate"]
            ).input_features[0]  # shape (N_MELS, N_FRAMES)
            ids = processor.tokenizer(text).input_ids
        except Exception as e:
            skipped += 1
            continue
        yield {"input_features": feats, "labels": ids}
        count += 1
        pbar.update(1)
    pbar.close()
    print(f"Done: {count} processed, {skipped} skipped")

ds_features = Features({
    "input_features": Array2D(shape=(N_MELS, N_FRAMES), dtype="float32"),
    "labels": Sequence(Value("int32")),
})

print(f"Streaming {MAX_SAMPLES} samples → disk...")
all_ds = Dataset.from_generator(
    sample_generator,
    features=ds_features,
    cache_dir=CACHE_DIR,
    writer_batch_size=100,
)
print(f"Dataset on disk: {len(all_ds)} samples")

eval_n   = max(50, int(len(all_ds) * EVAL_FRAC))
eval_ds  = all_ds.select(range(eval_n))
train_ds = all_ds.select(range(eval_n, len(all_ds)))
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

In [ ]:
# ── 8. Data collator + WER metric ─────────────────────────────────────────────
import evaluate
from dataclasses import dataclass
from typing import Any, List, Dict

wer_metric = evaluate.load("wer")

@dataclass
class WhisperCollator:
    processor: Any

    def __call__(self, features: List[Dict]) -> Dict:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Strip leading BOS token added by tokenizer
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

print("Collator and metric ready")

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────────
import math
from pathlib import Path
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Eval/save twice per epoch so we get checkpoints before session expires
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
eval_steps = max(25, steps_per_epoch // 2)
print(f"Steps per epoch: {steps_per_epoch}  |  eval/save every: {eval_steps} steps")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    logging_steps=10,
    report_to=[],
    remove_unused_columns=False,  # PEFT requires this
    label_names=["labels"],
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=WhisperCollator(processor),
    compute_metrics=compute_metrics,
    # tokenizer argument removed in transformers>=4.46 — WhisperCollator handles padding
)

trainer.train()

In [ ]:
# ── 10. Save adapter + processor ──────────────────────────────────────────────
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")
print()
print("To load and use locally:")
print(f"  from peft import PeftModel")
print(f"  from transformers import WhisperForConditionalGeneration, WhisperProcessor")
print(f"  base = WhisperForConditionalGeneration.from_pretrained('{BASE_MODEL}')")
print(f"  model = PeftModel.from_pretrained(base, '{OUTPUT_DIR}')")
print(f"  processor = WhisperProcessor.from_pretrained('{OUTPUT_DIR}')")

In [ ]:
# ── 11. Quick inference test ──────────────────────────────────────────────────
# Runs one sample from the eval set through the fine-tuned model.
import torch

model.eval()
sample = eval_ds[0]

input_features = torch.tensor(sample["input_features"]).unsqueeze(0)
# Move to same device as model (handles device_map=auto)
device = next(p for p in model.parameters() if p.requires_grad).device
input_features = input_features.to(device).half()

with torch.no_grad():
    pred_ids = model.generate(
        input_features,
        language=LANGUAGE,
        task="transcribe",
        max_new_tokens=225,
    )

pred_text = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]
label_ids = [l if l != -100 else processor.tokenizer.pad_token_id for l in sample["labels"]]
label_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

print(f"Reference : {label_text}")
print(f"Prediction: {pred_text}")

In [ ]:
# ── 12. Push to HF Hub (optional) ─────────────────────────────────────────────
if HF_HUB_MODEL_ID:
    print(f"Pushing to https://huggingface.co/{HF_HUB_MODEL_ID} ...")
    model.push_to_hub(HF_HUB_MODEL_ID)
    processor.push_to_hub(HF_HUB_MODEL_ID)
    print("Done! Adapter is now public on HF Hub.")
else:
    print("HF_HUB_MODEL_ID is empty — skipping hub push.")
    print("Set it in the Config cell and re-run this cell to publish.")